# 08 — CAD Export

Export the best v2 design for manufacturing:
1. STL mesh (body + wing)
2. STEP v2 — watertight solid (ThruSections loft)
3. Airfoil profiles (.dat Selig format)
4. LE/TE splines for CAD loft
5. Assembly JSON for downstream pipeline
6. EDF duct positioning
7. STEP v3 — OML + propulsion duct (boolean integration)

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
from pathlib import Path

from src.parameterization.design_variables import BWBParams, params_from_vector
from src.parameterization.bwb_aircraft import (
    build_airplane, compute_wing_area, compute_aspect_ratio, compute_mac,
    compute_internal_volume, estimate_structural_mass,
)
from src.aero.evaluator import AeroEvaluator
from src.aero.mission import MissionCondition
from src.propulsion.edf_model import EDF_70MM, duct_fits_in_body

In [2]:
%load_ext autoreload
%autoreload 2

## 1. Load Best Design

In [3]:
try:
    best_x = np.load('../output/best_x_v2.npy')
    params = params_from_vector(best_x)
    print('Loaded v2 optimized design (real CG + elevon trim)')
except FileNotFoundError:
    try:
        best_x = np.load('../output/best_x.npy')
        params = params_from_vector(best_x)
        print('Loaded v1 design (fallback)')
    except FileNotFoundError:
        params = BWBParams()
        print('Using default params')

mission = MissionCondition()
result = AeroEvaluator(mission, use_cg=True).evaluate(params)

Loaded v2 optimized design (real CG + elevon trim)


## 2. Export STL

In [ ]:
from src.visualization.export import export_aircraft_stl, export_aircraft_step

out_dir = Path('../output/cad_export')
out_dir.mkdir(parents=True, exist_ok=True)

# STL (triangulated mesh, for visualization)
n_tri = export_aircraft_stl(params, str(out_dir / 'neuron_v2.stl'))
print(f'STL exported: {n_tri} triangles -> {out_dir}/neuron_v2.stl')

# STEP (watertight solid via ThruSections loft)
metrics = export_aircraft_step(params, str(out_dir / 'neuron_v2_solid.step'))
print(f'STEP exported: {metrics["file_size_kb"]:.0f} KB -> {metrics["path"]}')
print(f'  Valid: {metrics["is_valid"]}  Volume: {metrics["volume_mm3"]/1e3:.1f} cm3')

> **Note:** Use `export_aircraft_step(params, path, include_propulsion=True)` (in notebook 09) for the full compound with control surface reference geometry and propulsion duct.

## 3. Export CAD Profiles

In [5]:
from src.visualization.export import export_cad_profiles

assembly = export_cad_profiles(params, str(out_dir))
print(f'CAD profiles exported to {out_dir}/')
print(f'Sections: {len(assembly["sections"])}')

CAD profiles exported to ..\output\cad_export/
Sections: 9


## 4. EDF Positioning

In [6]:
print('═══ EDF Integration ═══')
body_chord = params.body_root_chord
body_height = params.body_tc_root * body_chord
print(f'  Body chord       : {body_chord*1000:.0f} mm')
print(f'  Body height (max): {body_height*1000:.1f} mm')
print(f'  EDF duct OD      : {EDF_70MM.duct_outer_diameter*1000:.0f} mm')
print(f'  Duct fits        : {duct_fits_in_body(params.body_tc_root, body_chord, EDF_70MM)}')
print(f'  EDF position X   : {body_chord*0.45*1000:.0f} mm from nose (45% chord)')
print(f'  Thrust axis Z    : 0 mm (aligned with CG)')

═══ EDF Integration ═══
  Body chord       : 741 mm
  Body height (max): 159.6 mm
  EDF duct OD      : 78 mm
  Duct fits        : True
  EDF position X   : 334 mm from nose (45% chord)
  Thrust axis Z    : 0 mm (aligned with CG)


## 5. Design Report

In [7]:
print('===================================================')
print('       nEUROn v2 -- Final Design Report')
print('===================================================')
print(f'\nGeometry:')
print(f'  Span             : {2*params.half_span:.2f} m')
print(f'  Wing root chord  : {params.wing_root_chord*1000:.0f} mm')
print(f'  Body chord       : {params.body_root_chord*1000:.0f} mm')
print(f'  Body width       : {2*params.body_halfwidth*1000:.0f} mm')
print(f'  Taper ratio      : {params.taper_ratio:.3f}')
print(f'  LE sweep (wing)  : {params.le_sweep_deg:.1f} deg')
print(f'  Wing area        : {compute_wing_area(params)*1e4:.0f} cm2')
print(f'  Aspect ratio     : {compute_aspect_ratio(params):.2f}')
print(f'\nAerodynamics:')
print(f'  L/D              : {result["L_over_D"]:.2f}')
print(f'  CL               : {result["CL"]:.4f}')
print(f'  CD               : {result["CD"]:.5f}')
print(f'  Static margin    : {result["static_margin"]*100:.1f}% MAC')
print(f'\nControl (v2):')
print(f'  CG x/c           : {result.get("x_cg_frac", 0):.3f}')
print(f'  Elevon deflection : {result.get("elevon_deflection", 0):.1f} deg')
print(f'  Cn_beta          : {result["Cn_beta"]:.4f}')
print(f'  Cl_beta          : {result.get("Cl_beta", 0):.4f}')
print(f'\nStructure:')
print(f'  Struct mass      : {result["struct_mass"]*1000:.0f} g')
print(f'  Internal volume  : {result["internal_volume"]*1e6:.0f} cm3')
print(f'\nPropulsion:')
print(f'  T/D              : {result["T_over_D"]:.2f}')
print(f'  Endurance        : {result["endurance_min"]:.1f} min')
print(f'  Range            : {result["range_km"]:.1f} km')
print(f'\nFeasible: {result["is_feasible"]}')

       nEUROn v2 -- Final Design Report

Geometry:
  Span             : 1.51 m
  Wing root chord  : 449 mm
  Body chord       : 741 mm
  Body width       : 160 mm
  Taper ratio      : 0.290
  LE sweep (wing)  : 33.2 deg
  Wing area        : 4860 cm2
  Aspect ratio     : 4.68

Aerodynamics:
  L/D              : 9.66
  CL               : 0.1026
  CD               : 0.01062
  Static margin    : 5.4% MAC

Control (v2):
  CG x/c           : 0.447
  Elevon deflection : 3.6 deg
  Cn_beta          : 0.0057
  Cl_beta          : -0.0234

Structure:
  Struct mass      : 793 g
  Internal volume  : 2532 cm3

Propulsion:
  T/D              : 2.74
  Endurance        : 14.0 min
  Range            : 16.8 km

Feasible: True
